# 119 — MCP Server Builder
## What you'll learn: how to build a stdio MCP server and expose typed tools to AI clients
⏱ ~60 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/119-mcp-server-builder/mcp_server_builder_workbook.ipynb)

The **Model Context Protocol (MCP)** is an open standard that lets AI assistants like Claude call tools hosted on external servers. Instead of hardcoding tools into every agent, you build an MCP server once and any compatible client — Claude Desktop, LangGraph agents, custom bots — can discover and call your tools automatically.

This workshop builds a real, runnable stdio MCP server from scratch. You'll implement 3 domain tools, define their JSON Schema types, and connect the server to Claude Desktop.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — MCP protocol, JSON-RPC 2.0, stdio vs SSE transport |
| 2 | **Tool schemas** — JSON Schema definitions for MCP tools |
| 3 | **Domain tools** — weather lookup, knowledge base search, unit conversion |
| 4 | **MCP server** — Server class, @list_tools, @call_tool decorators |
| 5 | **Connecting a client** — Claude Desktop config and LangGraph MCP client |
| 6 | **Testing tools directly** — run all 3 tools as plain Python |
| 7 | **Transport comparison** — stdio vs SSE |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- `langchain-openai`, `python-dotenv`, `mcp` (optional — demo mode works without it)

## Setup — Install Dependencies

In [ ]:
# Install required packages
%pip install -q langchain-openai python-dotenv mcp==1.26.0
print('Dependencies installed.')

## API Key Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Colab: use userdata secrets
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('API key loaded from Colab secrets.')
except Exception:
    if os.environ.get('OPENAI_API_KEY'):
        print('API key loaded from .env')
    else:
        print('WARNING: OPENAI_API_KEY not set. Set it before running LLM cells.')

## Part 1 — Concepts: What is MCP?

### The Model Context Protocol

MCP is an open protocol by Anthropic that standardizes how AI clients discover and call external tools. Instead of every agent framework inventing its own tool-calling convention, MCP provides:

- A **server** that exposes named tools with JSON Schema input definitions
- A **client** (Claude Desktop, LangGraph, etc.) that discovers tools and calls them
- A **transport layer** (stdio or SSE) that handles the communication

### JSON-RPC 2.0 under the hood

All MCP messages are JSON-RPC 2.0 envelopes:

```json
// Client → Server: call a tool
{"jsonrpc": "2.0", "id": 1, "method": "tools/call",
 "params": {"name": "get_weather", "arguments": {"city": "London"}}}

// Server → Client: result
{"jsonrpc": "2.0", "id": 1,
 "result": {"content": [{"type": "text", "text": "{...}"}]}}
```

You don't write this JSON manually — the MCP SDK handles it. But knowing the wire format helps when debugging.

### Two transport options

| Transport | How it works | Best for |
|-----------|-------------|----------|
| **stdio** | JSON piped through stdin/stdout | Local servers, Claude Desktop, CLI tools |
| **SSE** | HTTP Server-Sent Events | Remote servers, browser clients, multi-user deployments |

This example uses **stdio** — it's the simplest and works with Claude Desktop out of the box.

## Part 2 — Tool Schemas

Every MCP tool must declare its name, description, and input schema in **JSON Schema** format. The client uses this schema to:
- Show the user what tools are available
- Validate arguments before calling
- Let the LLM know what parameters to provide

Here's the schema structure:

In [ ]:
import json

# Example: a minimal MCP tool schema
example_schema = {
    "name": "get_weather",
    "description": "Get current weather data for a city.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "City name (e.g., 'London', 'Tokyo')"
            }
        },
        "required": ["city"]
    }
}

print(json.dumps(example_schema, indent=2))

## Part 3 — Domain Tools

We'll implement 3 tools as plain Python functions first, then wire them into an MCP server.

### Tool 1: `get_weather`

In [ ]:
WEATHER_DATA = {
    "london": {"temp_c": 15, "condition": "cloudy", "humidity": 72},
    "new york": {"temp_c": 22, "condition": "sunny", "humidity": 55},
    "tokyo": {"temp_c": 28, "condition": "partly cloudy", "humidity": 80},
    "paris": {"temp_c": 18, "condition": "rainy", "humidity": 85},
    "sydney": {"temp_c": 20, "condition": "clear", "humidity": 60},
}

def get_weather(city: str) -> dict:
    """Return mock weather data for a city."""
    key = city.lower().strip()
    data = WEATHER_DATA.get(key, {"temp_c": 20, "condition": "unknown", "humidity": 50})
    return {
        "city": city,
        "temperature_c": data["temp_c"],
        "temperature_f": round(data["temp_c"] * 9 / 5 + 32, 1),
        "condition": data["condition"],
        "humidity_pct": data["humidity"],
    }

# Test it
for city in ["London", "Tokyo", "Sydney", "Miami"]:
    w = get_weather(city)
    print(f"{w['city']}: {w['temperature_c']}°C / {w['temperature_f']}°F, {w['condition']}")

### Tool 2: `search_knowledge_base`

In [ ]:
KNOWLEDGE_BASE = [
    "Python was created by Guido van Rossum in 1991.",
    "LangGraph is a library for building stateful multi-actor applications with LLMs.",
    "The MCP protocol uses JSON-RPC 2.0 for communication between clients and servers.",
    "Stdio transport pipes JSON messages through stdin/stdout.",
    "SSE (Server-Sent Events) transport uses HTTP for browser-compatible MCP servers.",
    "Tool schemas in MCP are defined using JSON Schema.",
    "Claude Desktop supports MCP servers via its config file at ~/Library/Application Support/Claude/claude_desktop_config.json.",
    "LangGraph nodes are Python functions that receive and return state dicts.",
    "OpenAI's gpt-5.4-nano is a cost-efficient model for tool-calling tasks.",
    "Pydantic v2 is the recommended validation library for LangChain/LangGraph projects.",
]

def search_knowledge_base(query: str) -> list[str]:
    """Search the hardcoded knowledge base for facts matching the query."""
    query_lower = query.lower()
    matches = [
        fact for fact in KNOWLEDGE_BASE
        if any(word in fact.lower() for word in query_lower.split())
    ]
    return matches or ["No matching facts found."]

# Test it
for query in ["MCP protocol", "LangGraph nodes", "Claude Desktop"]:
    results = search_knowledge_base(query)
    print(f"Query: '{query}'")
    for r in results:
        print(f"  - {r}")

### Tool 3: `convert_units`

In [ ]:
UNIT_CONVERSIONS = {
    ("km", "miles"): 0.621371,
    ("miles", "km"): 1.60934,
    ("kg", "lbs"): 2.20462,
    ("lbs", "kg"): 0.453592,
    ("c", "f"): None,  # special case
    ("f", "c"): None,  # special case
}

def convert_units(value: float, from_unit: str, to_unit: str) -> dict:
    """Convert between km/miles, kg/lbs, or C/F."""
    from_u = from_unit.lower().strip()
    to_u = to_unit.lower().strip()

    if from_u == "c" and to_u == "f":
        result = value * 9 / 5 + 32
    elif from_u == "f" and to_u == "c":
        result = (value - 32) * 5 / 9
    else:
        factor = UNIT_CONVERSIONS.get((from_u, to_u))
        if factor is None:
            return {"error": f"Unsupported conversion: {from_unit} to {to_unit}"}
        result = value * factor

    return {
        "input": value,
        "from_unit": from_unit,
        "to_unit": to_unit,
        "result": round(result, 4),
    }

# Test it
conversions = [
    (100, "km", "miles"),
    (70, "kg", "lbs"),
    (100, "C", "F"),
    (212, "F", "C"),
]
for val, frm, to in conversions:
    r = convert_units(val, frm, to)
    print(f"  {r['input']} {r['from_unit']} = {r['result']} {r['to_unit']}")

## Part 4 — Building the MCP Server

The `mcp` Python SDK provides a `Server` class that handles the JSON-RPC protocol. You register handlers with decorators:

- `@app.list_tools()` — called when a client asks what tools are available
- `@app.call_tool()` — called when a client invokes a specific tool

Here's the server setup (without the asyncio event loop — we can't `asyncio.run()` inside Colab):

In [ ]:
import json

# Check if mcp is installed
try:
    from mcp.server import Server
    from mcp.server.stdio import stdio_server
    from mcp.types import TextContent, Tool
    HAS_MCP = True
    print("mcp SDK is installed — server mode available.")
except ImportError:
    HAS_MCP = False
    print("mcp SDK not installed — showing structure only.")
    print("Install with: pip install mcp")

# Tool schemas (MCP-compatible JSON Schema format)
TOOL_SCHEMAS = [
    {
        "name": "get_weather",
        "description": "Get current weather data for a city.",
        "inputSchema": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "City name"}},
            "required": ["city"],
        },
    },
    {
        "name": "search_knowledge_base",
        "description": "Search a knowledge base of facts about Python, LLMs, and MCP.",
        "inputSchema": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "Search query"}},
            "required": ["query"],
        },
    },
    {
        "name": "convert_units",
        "description": "Convert between km/miles, kg/lbs, or Celsius/Fahrenheit.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "value": {"type": "number"},
                "from_unit": {"type": "string", "description": "km, miles, kg, lbs, C, or F"},
                "to_unit": {"type": "string", "description": "km, miles, kg, lbs, C, or F"},
            },
            "required": ["value", "from_unit", "to_unit"],
        },
    },
]

print(f"\nDefined {len(TOOL_SCHEMAS)} tools:")
for s in TOOL_SCHEMAS:
    print(f"  {s['name']}: {s['description']}")

In [ ]:
# Show the full server structure (conceptual — not run with asyncio.run() in Colab)
server_code = '''
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import TextContent, Tool
import asyncio, json, sys

app = Server("domain-tools")

@app.list_tools()
async def list_tools():
    # Return Tool objects for every tool the server exposes
    return [
        Tool(name=s["name"], description=s["description"], inputSchema=s["inputSchema"])
        for s in TOOL_SCHEMAS
    ]

@app.call_tool()
async def call_tool(name: str, arguments: dict):
    # Dispatch to the correct Python function
    if name == "get_weather":
        result = get_weather(arguments["city"])
    elif name == "search_knowledge_base":
        result = search_knowledge_base(arguments["query"])
    elif name == "convert_units":
        result = convert_units(
            arguments["value"], arguments["from_unit"], arguments["to_unit"]
        )
    else:
        result = {"error": f"Unknown tool: {name}"}
    # Return as MCP TextContent
    return [TextContent(type="text", text=json.dumps(result, indent=2))]

# Start the stdio server
async def serve():
    async with stdio_server() as (read_stream, write_stream):
        await app.run(read_stream, write_stream, app.create_initialization_options())

print("[MCP] domain-tools server starting on stdio...", file=sys.stderr)
asyncio.run(serve())
'''

print(server_code)

## Part 5 — Connecting a Client

### Claude Desktop

Claude Desktop reads its MCP server list from a JSON config file:

**macOS:** `~/Library/Application Support/Claude/claude_desktop_config.json`  
**Windows:** `%APPDATA%\Claude\claude_desktop_config.json`

Add your server like this:

```json
{
  "mcpServers": {
    "domain-tools": {
      "command": "python",
      "args": ["/absolute/path/to/examples/119-mcp-server-builder/main.py"]
    }
  }
}
```

After saving and restarting Claude Desktop, you'll see the 3 tools listed in the tool picker.

### LangGraph MCP Client Pattern

LangGraph agents can call MCP servers as tools using the `langchain-mcp-adapters` package:

In [ ]:
# Pattern: how a LangGraph agent would call our MCP server
# (requires: pip install langchain-mcp-adapters)

langgraph_mcp_pattern = '''
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4-nano")

# Point at our server
server_params = StdioServerParameters(
    command="python",
    args=["examples/119-mcp-server-builder/main.py"]
)

async def run_agent():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            # Load tools from MCP server as LangChain-compatible tools
            tools = await load_mcp_tools(session)
            agent = create_react_agent(llm, tools)
            result = await agent.ainvoke({"messages": [
                {"role": "user", "content": "What is 100km in miles? Also get weather for Tokyo."}
            ]})
            return result
'''

print(langgraph_mcp_pattern)

# Verify the real stdio transport without requiring an LLM or LangGraph.
import json
import sys
from pathlib import Path
from mcp import ClientSession
from mcp.client.stdio import StdioServerParameters, stdio_client

async def verify_mcp_round_trip():
    example_dir = Path.cwd()
    if not (example_dir / "main.py").exists():
        example_dir = example_dir / "examples/119-mcp-server-builder"
    params = StdioServerParameters(
        command=sys.executable,
        args=["main.py"],
        cwd=str(example_dir),
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            names = [tool.name for tool in tools.tools]
            assert names == ["get_weather", "search_knowledge_base", "convert_units"]
            response = await session.call_tool(
                "convert_units",
                arguments={"value": 100, "from_unit": "km", "to_unit": "miles"},
            )
            result = json.loads(response.content[0].text)
            assert result["result"] == 62.1371
            return names, result

tool_names, conversion = await verify_mcp_round_trip()
print(f"MCP stdio round trip passed: {tool_names}; 100 km = {conversion['result']} miles")

## Part 6 — Testing Tools Directly

You don't need the MCP server running to test your tools. Run them as plain Python functions first — faster feedback loop, easier to debug.

In [ ]:
print("=== Running all 3 tools directly ===\n")

# Tool 1: Weather
print("--- get_weather ---")
for city in ["London", "Tokyo", "Paris", "Sydney", "New York"]:
    w = get_weather(city)
    print(f"  {w['city']:12} {w['temperature_c']}°C / {w['temperature_f']}°F  {w['condition']}  humidity: {w['humidity_pct']}%")

# Tool 2: Knowledge base
print("\n--- search_knowledge_base ---")
queries = ["MCP", "LangGraph", "gpt-5.4-nano", "Pydantic"]
for q in queries:
    hits = search_knowledge_base(q)
    print(f"  '{q}': {len(hits)} result(s)")
    for h in hits:
        print(f"    {h}")

# Tool 3: Unit conversion
print("\n--- convert_units ---")
tests = [
    (100, "km", "miles"),
    (26.2, "miles", "km"),
    (70, "kg", "lbs"),
    (154, "lbs", "kg"),
    (100, "C", "F"),
    (32, "F", "C"),
    (5, "cups", "liters"),  # unsupported
]
for val, frm, to in tests:
    r = convert_units(val, frm, to)
    if "error" in r:
        print(f"  {val} {frm} -> {to}: ERROR: {r['error']}")
    else:
        print(f"  {r['input']} {r['from_unit']} = {r['result']} {r['to_unit']}")

## Part 7 — Transport Comparison: stdio vs SSE

| Feature | stdio | SSE (HTTP) |
|---------|-------|------------|
| **How it works** | JSON piped through stdin/stdout | HTTP long-polling with server-sent events |
| **Setup** | Zero — just run the Python script | Needs an HTTP server (FastAPI, etc.) |
| **Best for** | Local tools, Claude Desktop, CLI | Remote servers, browser clients, multi-user |
| **Security** | Process isolation, no network exposure | Needs auth (API keys, JWT) |
| **Debugging** | Can log to stderr, pipe manually | Use HTTP inspector, browser devtools |
| **Scaling** | Single user per process | Multiple clients, stateless HTTP |

### When to use stdio
- Building a local productivity tool for Claude Desktop
- You want zero infrastructure (no server, no port, no auth)
- The tool accesses local files, databases, or system APIs

### When to use SSE
- Multiple users need to share the same tool server
- The tool is deployed remotely (cloud, VPS)
- You need browser clients to call the server

This example uses **stdio** — the right choice for a local demo server.

To switch to SSE, replace `asyncio.run(stdio_server(app))` with the SSE transport from the `mcp` SDK:

```python
from mcp.server.sse import SseServerTransport
# ... mount on a FastAPI/Starlette app
```

## Exercises

### Exercise 1: Add a 4th tool — `get_time`

Add a `get_time(timezone: str) -> dict` tool that returns the current time in a given timezone.

Requirements:
- Use only the stdlib `datetime` module
- Support at least: UTC, US/Eastern, US/Pacific, Europe/London
- Return a dict with `timezone`, `datetime`, and `utc_offset` keys
- Write its JSON Schema definition

Hint: use `datetime.timezone` or the `zoneinfo` module (Python 3.9+).

In [ ]:
# Your solution here
from datetime import datetime, timezone, timedelta

def get_time(tz_name: str) -> dict:
    # YOUR CODE HERE
    pass

# Test:
# print(get_time("UTC"))
# print(get_time("US/Eastern"))

### Exercise 2: Add more unit conversions

Extend `UNIT_CONVERSIONS` and `convert_units` to support:
- meters ↔ feet
- liters ↔ gallons (US)
- grams ↔ ounces

Then update the `convert_units` JSON Schema description to list the new units.

In [ ]:
# Your solution here
EXTENDED_CONVERSIONS = {
    **UNIT_CONVERSIONS,
    # Add new conversions:
    ("meters", "feet"): None,   # fill in the factor
    ("feet", "meters"): None,
    ("liters", "gallons"): None,
    ("gallons", "liters"): None,
    ("grams", "ounces"): None,
    ("ounces", "grams"): None,
}

# Test:
# r = convert_units(1.8, "meters", "feet")
# print(r)

## Answer Key

In [ ]:
# Answer: Exercise 1 — get_time tool
from datetime import datetime, timezone, timedelta

TIMEZONE_OFFSETS = {
    "UTC": 0,
    "US/Eastern": -5,
    "US/Pacific": -8,
    "Europe/London": 0,
    "Asia/Tokyo": 9,
    "Australia/Sydney": 10,
}

def get_time(tz_name: str) -> dict:
    """Return current time in the given timezone."""
    offset_hours = TIMEZONE_OFFSETS.get(tz_name, 0)
    tz = timezone(timedelta(hours=offset_hours))
    now = datetime.now(tz)
    return {
        "timezone": tz_name,
        "datetime": now.strftime("%Y-%m-%d %H:%M:%S"),
        "utc_offset": f"UTC{offset_hours:+d}",
    }

# JSON Schema for the tool
get_time_schema = {
    "name": "get_time",
    "description": "Get the current date and time in a given timezone.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "timezone": {
                "type": "string",
                "description": "Timezone name (e.g., UTC, US/Eastern, US/Pacific, Asia/Tokyo)"
            }
        },
        "required": ["timezone"]
    }
}

for tz in ["UTC", "US/Eastern", "Asia/Tokyo"]:
    print(get_time(tz))

In [ ]:
# Answer: Exercise 2 — extended unit conversions
EXTENDED_CONVERSIONS = {
    **UNIT_CONVERSIONS,
    ("meters", "feet"): 3.28084,
    ("feet", "meters"): 0.3048,
    ("liters", "gallons"): 0.264172,
    ("gallons", "liters"): 3.78541,
    ("grams", "ounces"): 0.035274,
    ("ounces", "grams"): 28.3495,
}

def convert_units_extended(value: float, from_unit: str, to_unit: str) -> dict:
    """Convert between a wide range of units."""
    from_u = from_unit.lower().strip()
    to_u = to_unit.lower().strip()

    if from_u == "c" and to_u == "f":
        result = value * 9 / 5 + 32
    elif from_u == "f" and to_u == "c":
        result = (value - 32) * 5 / 9
    else:
        factor = EXTENDED_CONVERSIONS.get((from_u, to_u))
        if factor is None:
            return {"error": f"Unsupported conversion: {from_unit} to {to_unit}"}
        result = value * factor

    return {"input": value, "from_unit": from_unit, "to_unit": to_unit, "result": round(result, 4)}

# Tests
for val, frm, to in [(1.8, "meters", "feet"), (1, "liters", "gallons"), (100, "grams", "ounces")]:
    r = convert_units_extended(val, frm, to)
    print(f"  {r['input']} {r['from_unit']} = {r['result']} {r['to_unit']}")

## Workshop Complete

You've built a fully functional stdio MCP server with 3 typed domain tools.

### What you built
- `get_weather` — city weather lookup with Celsius/Fahrenheit
- `search_knowledge_base` — keyword search over a fact store
- `convert_units` — unit conversion for km/miles, kg/lbs, C/F
- JSON Schema definitions for all 3 tools
- A stdio MCP server using the `mcp` Python SDK
- Claude Desktop configuration to connect it

### What to try next
- Add the `get_time` tool from Exercise 1 to the real server
- Swap the mock `WEATHER_DATA` for a real API call (e.g., Open-Meteo — free, no key required)
- Replace `KNOWLEDGE_BASE` with a vector search over your own documents
- Try the SSE transport to make the server accessible over HTTP
- Use `langchain-mcp-adapters` to connect a LangGraph agent to your server

### Key concepts
- MCP uses JSON-RPC 2.0 — tools are just named functions with JSON Schema types
- stdio transport is zero-infrastructure — perfect for local tools
- Any MCP-compatible client (Claude Desktop, LangGraph, etc.) can call your server without code changes
- Test tools as plain Python functions first, then wire them into the MCP layer